In [1]:
import csv
import numpy as np
import random
from encoding import *
from pyDOE2 import lhs
import copy
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pickle
import os
import copy 
import json
import csv
import numpy as np
import pandas as pd
import tensorflow as tf
import torchaudio
import torch
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from keras.models import Sequential, Model
from keras.layers import Resizing, Conv2D, Dropout, BatchNormalization, MaxPooling2D, MaxPool2D, Flatten, Dense, Input, LeakyReLU
from tqdm import tqdm
import tensorflow_addons as tfa
from latin import *
from audio import *
import tqdm
import numpy as np
import pandas as pd
import tensorflow as tf
import torchaudio
import torch
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import copy
import os



c:\Users\herna\anaconda3\envs\tf\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
c:\Users\herna\anaconda3\envs\tf\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.10.1 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want

In [2]:


def int_to_real_dom(num, domain):
  min_i, max_i = domain
  r = (num - min_i) / (max_i - min_i)
  return r

def real_to_int_dom(num, domain):
  min_i, max_i = domain
  value = min_i + num * (max_i - min_i)
  if isinstance(min_i, int) and isinstance(max_i, int):
      value = int(round(value))
  return value

def convert_individual(ind, to_real=True):
    real_rep = []
    N = max(layer_type_options.keys())
    for i in range(0, len(ind), 4):
        layer_type_idx = ind[i]
        domain_layer_type = [0, N]
        if to_real:
            real_rep.append(int_to_real_dom(layer_type_idx, domain_layer_type))
            layer_type = layer_type_options.get(layer_type_idx, 'DontCare')
        else:
            real_rep.append(real_to_int_dom(layer_type_idx, domain_layer_type))
            layer_type = layer_type_options.get(real_rep[i], 'DontCare')

        # Decode based on layer type
        if layer_type in ['Conv2D', 'DepthwiseConv2D']:
            if to_real:
                real_rep.append(int_to_real_dom(ind[i + 1], [4, 32]))
                real_rep.append(int_to_real_dom(ind[i + 2], [0, 1]))
                real_rep.append(int_to_real_dom(ind[i + 3], [0, 3]))
            else:
                real_rep.append(real_to_int_dom(ind[i + 1], [4, 32]))
                real_rep.append(real_to_int_dom(ind[i + 2], [0, 1]))
                real_rep.append(real_to_int_dom(ind[i + 3], [0, 3]))
        elif layer_type == 'BatchNorm':
            real_rep.extend([0, 0, 0])
        elif layer_type == 'MaxPooling':
            if to_real:
                real_rep.append(int_to_real_dom(ind[i + 1], [0, 1]))
            else:
                real_rep.append(real_to_int_dom(ind[i + 1], [0, 1]))
            real_rep.extend([0, 0])
        elif layer_type == 'Dropout':
            if to_real:
                real_rep.append(int_to_real_dom(ind[i + 1], [0, 3]))
            else:
                real_rep.append(real_to_int_dom(ind[i + 1], [0, 3]))
            real_rep.extend([0, 0])
        elif layer_type == 'Dense':
            if to_real:
                real_rep.append(int_to_real_dom(ind[i + 1], [1, 512]))
                real_rep.append(int_to_real_dom(ind[i + 2], [0, 3]))
            else:
                real_rep.append(real_to_int_dom(ind[i + 1], [1, 512]))
                real_rep.append(real_to_int_dom(ind[i + 2], [0, 3]))
            real_rep.append(0)
        elif layer_type == 'Flatten':
            real_rep.extend([0, 0, 0])
        elif layer_type == 'Repetition':
            if to_real:
                real_rep.append(int_to_real_dom(ind[i + 1], [1, 4]))
                real_rep.append(int_to_real_dom(ind[i + 2], [1, 32]))
            else:
                real_rep.append(real_to_int_dom(ind[i + 1], [1, 4]))
                real_rep.append(real_to_int_dom(ind[i + 2], [1, 32]))
            real_rep.append(0)
        elif layer_type == 'DontCare':
            real_rep.extend([0, 0, 0])
    return real_rep


def get_succ_m(parents, children):
  # Count if mutation was successful than the average parent fitness
  parent_fitness = [parent['fitness'] for parent in parents]
  avg_fitness = np.mean(parent_fitness)

  succ_m_count = sum(1 for child in children if child['fitness'] < avg_fitness)

  return succ_m_count

def get_cr_points(rp, n):
  indexes = list(range(n))
  j_star = random.sample(indexes, n // 2)

  for j in range(n):
    if random.random() < rp and j not in j_star:
      j_star.append(j)

  return j_star

def pop_gen(num_models, max_alleles=48):
    """
    Genera una población inicial utilizando el hipercubo latino y las funciones existentes.

    Args:
        num_models: int - Número de individuos a generar.
        max_alleles: int - Número máximo de alelos en los cromosomas.

    Returns:
        list - Lista de diccionarios con individuos y su fitness inicializado a 0.
    """
    archs = []
    dimensions = 12 * 3  # 12 capas, 3 parámetros por capa

    # Generar muestras del hipercubo latino
    latin_samples = generate_latin_hypercube_samples(num_models, dimensions)

    for sample in latin_samples:
        # Transformar cada muestra en una arquitectura
        model_samples = sample.reshape(12, 3)
        model_dict = {
            "layers": [map_to_architecture_params(layer_sample) for layer_sample in model_samples]
        }

        # Codificar el modelo y repararlo
        encoded_chromosome = encode_model_architecture(model_dict, max_alleles=max_alleles)
        repaired_architecture = fixArch(encoded_chromosome)

        # Añadir a la población
        archs.append({'individual': repaired_architecture, 'fitness': 0})

    return archs


def pop_gen_with_initial_parents(num_models, initial_parents, max_alleles=48):
    """
    Genera una población inicial combinando padres predefinidos con una población generada aleatoriamente.

    Args:
        num_models (int): Número total de individuos en la población inicial.
        initial_parents (list): Lista de cromosomas predefinidos como padres iniciales.
        max_alleles (int): Número máximo de alelos en los cromosomas.

    Returns:
        list: Población inicial con individuos y su fitness inicializado a 0.
    """
    # Generar población aleatoria restante
    num_random = num_models - len(initial_parents)
    if num_random < 0:
        raise ValueError("El número de padres iniciales excede el tamaño de la población deseada.")

    # Generar población aleatoria usando el hipercubo latino
    dimensions = max_alleles
    latin_samples = generate_latin_hypercube_samples(num_random, dimensions)

    # Convertir las muestras aleatorias en cromosomas válidos
    random_individuals = []
    for sample in latin_samples:
        repaired = fixArch(sample.tolist())
        random_individuals.append({'individual': repaired, 'fitness': 0})

    # Incluir los padres iniciales en la población
    parent_individuals = [{'individual': fixArch(parent), 'fitness': 0} for parent in initial_parents]

    # Combinar padres iniciales y población aleatoria
    population = parent_individuals + random_individuals

    return population

def save_checkpoint(filename, population, generation, best_fitness_per_gen, Fs):
    """
    Guarda el estado del algoritmo evolutivo en un archivo.
    
    Args:
        filename (str): Nombre del archivo para guardar el checkpoint.
        population (list): Población actual.
        generation (int): Generación actual.
        best_fitness_per_gen (list): Mejor fitness por generación.
        Fs (list): Valores históricos del factor F (si se usa auto-adaptación).
    """
    checkpoint = {
        'population': population,
        'generation': generation,
        'best_fitness_per_gen': best_fitness_per_gen,
        'Fs': Fs
    }
    #print(checkpoint)
    with open(filename, 'wb') as f:
        pickle.dump(checkpoint, f)
    #print(f"Checkpoint guardado en {filename}")

def load_checkpoint(filename):
    """
    Carga el estado del algoritmo evolutivo desde un archivo.

    Args:
        filename (str): Nombre del archivo del checkpoint.

    Returns:
        dict: Estado del algoritmo evolutivo cargado.
    """
    if os.path.exists(filename):
        with open(filename, 'rb') as f:
            checkpoint = pickle.load(f)
        print(f"Checkpoint cargado desde {filename}")
        return checkpoint
    else:
        print(f"No se encontró un checkpoint previo en {filename}. Comenzando desde cero.")
        return None


def es(target_func, model_path, mu=10, lamb=1, F=2, rp=0.5, gens=100, n=10, auto_adapt=False, 
       initial_parents=None, checkpoint_file='checkpoint.pkl', resume=False):
    """
    Estrategia evolutiva con soporte para checkpoints.

    Args:
        target_func: Función objetivo.
        model_path: Ruta al modelo surrogado.
        mu: Tamaño de la población.
        lamb: Número de hijos generados por generación.
        F: Factor de escalamiento para mutación.
        rp: Probabilidad de recombinación.
        gens: Número total de generaciones.
        n: Ventana para auto-adaptación.
        auto_adapt: Si se usa auto-adaptación.
        initial_parents: Padres iniciales predefinidos.
        checkpoint_file: Archivo para guardar/cargar checkpoints.
        resume: Si se reanuda desde un checkpoint.
    """
    if resume:
        checkpoint = load_checkpoint(checkpoint_file)
        if checkpoint:
            pop = checkpoint['population']
            start_gen = checkpoint['generation']
            best_fitness_per_gen = checkpoint['best_fitness_per_gen']
            Fs = checkpoint['Fs']
        else:
            if initial_parents:
                pop = pop_gen_with_initial_parents(mu, initial_parents)
            else:
                pop = pop_gen(mu)       
            start_gen = 0
            best_fitness_per_gen = []
            Fs = []
    else:
        if initial_parents:
            pop = pop_gen_with_initial_parents(mu, initial_parents)
        else:
            pop = pop_gen(mu)
        start_gen = 0
        best_fitness_per_gen = []
        Fs = []

    succ_m_count = 0

    with tqdm(total=gens, initial=start_gen, desc="Generations", leave=False) as pbar_gens:
        for gen in range(start_gen, gens):
            children = []

            # Evaluar fitness de los padres usando el modelo surrogado
            for parent in pop:
                parent['fitness'] = target_func(parent['individual'], model_path)
            best_parent = max(pop, key=lambda x: x['fitness'])

            for i in range(lamb):
                parent = pop[i]['individual']
                parent_unit = convert_individual(parent)
                best_parent_c = convert_individual(best_parent['individual'])

                x2, x3 = random.sample(pop, 2)
                x2 = convert_individual(x2['individual'])
                x3 = convert_individual(x3['individual'])
                u = [best_parent_c[j] + F * (x2[j] - x3[j]) for j in range(len(best_parent_c))]
                u = fixArch(convert_individual(u, to_real=False))
                u = convert_individual(u)
                cr_points = get_cr_points(rp, len(parent))
                child = [u[j] if j in cr_points else parent[j] for j in range(len(u))]
                child = fixArch(convert_individual(child, to_real=False))
                children.append({'individual': child, 'fitness': 0})

            # Evaluar fitness de los hijos usando el modelo surrogado
            children = [{'individual': child['individual'], 'fitness': target_func(child['individual'], model_path)} 
                       for child in children]

            complete_pop = pop + children
            pop = sorted(complete_pop, key=lambda x: x['fitness'], reverse=True)[:mu]
            best_fitness_per_gen.append(max(pop, key=lambda x: x['fitness'])['fitness'])

            if auto_adapt:
                succ_m_count += get_succ_m(pop, children)
                if (gen + 1) % n == 0:
                    Fs.append(F)
                    ps = succ_m_count / (n * lamb)
                    F = F / 0.817 if ps > 1/5 else F * 0.817
                    succ_m_count = 0

            save_checkpoint(checkpoint_file, pop, gen + 1, best_fitness_per_gen, Fs)
            pbar_gens.update(1)

    best_element = max(pop, key=lambda x: x['fitness'])
    return best_element, best_fitness_per_gen, Fs


def eval_arch(ind, model_path):
    ind_c = copy.deepcopy(ind)
    reshaped_ind_c = np.array(ind_c).reshape(1, -1)
    model = joblib.load(model_path)
    acc = model.predict(reshaped_ind_c)
    return acc[0]


# Función para cargar supracheckpoint
def load_super_checkpoint(super_checkpoint_file):
    if os.path.exists(super_checkpoint_file):
        with open(super_checkpoint_file, 'rb') as f:
            super_checkpoint = pickle.load(f)
        print(f"Supracheckpoint cargado desde {super_checkpoint_file}")
        return super_checkpoint
    else:
        print(f"No se encontró un supracheckpoint en {super_checkpoint_file}. Comenzando desde el experimento 0.")
        return {'last_experiment': 0}

# Guardar supracheckpoint
def save_super_checkpoint(super_checkpoint_file, last_experiment):
    super_checkpoint = {'last_experiment': last_experiment}
    with open(super_checkpoint_file, 'wb') as f:
        pickle.dump(super_checkpoint, f)
    print(f"Supracheckpoint guardado en {super_checkpoint_file}")

# Archivo de supracheckpoint
super_checkpoint_file = 'super_checkpoint.pkl'
super_checkpoint = load_super_checkpoint(super_checkpoint_file)
last_experiment = super_checkpoint['last_experiment']

# Experimentos
from tqdm import tqdm
initial_parents = [
    [0, 30, 0, 0, 3, 0, 0, 0, 1, 0, 0, 0, 2, 1, 0, 0, 0, 16, 0, 0, 3, 0, 0, 0, 1, 0, 0, 0, 2, 1, 0, 0, 5, 0, 0, 0, 4, 256, 0, 0, 3, 1, 0, 0, 4, 1, 2, 0],
    [1, 0, 0, 0, 0, 16, 0, 1, 1, 0, 0, 0, 0, 8, 0, 1, 1, 0, 0, 0, 5, 0, 0, 0, 4, 32, 1, 0, 4, 1, 2, 0, 7, 0, 0, 0, 7, 0, 0, 0, 7, 0, 0, 0, 7, 0, 0, 0],
    [0, 32, 0, 1, 1, 0, 0, 0, 2, 1, 0, 0, 8, 3, 31, 0, 5, 0, 0, 0, 4, 256, 0, 0, 3, 3, 0, 0, 4, 1, 2, 0, 7, 0, 0, 0, 7, 0, 0, 0, 7, 0, 0, 0, 7, 0, 0, 0]
]

best_models = []
all_best_fitness = []
F_arr = []
NUMBER_EXPERIMENTS = 30
model_path = "../surrogates/F1_SVM_optimized.pkl"  # Puedes cambiar esta ruta según necesites


# Retomar experimentos desde el último registrado
for i in tqdm(range(last_experiment, NUMBER_EXPERIMENTS), desc="Running Experiments"):
    best_model, best_fitness_per_gen, Fs = es(
        eval_arch,
        model_path=model_path,
        gens=2000,  # 1000
        F=0.5,
        mu=1000,  # 5000
        auto_adapt=True,
        initial_parents=initial_parents,
        checkpoint_file=f"checkpoint_{i}.pkl",
        resume=True
    )
    F_arr.append(Fs)
    best_models.append(best_model)
    all_best_fitness.append(best_fitness_per_gen)

    # Actualizar y guardar supracheckpoint
    save_super_checkpoint(super_checkpoint_file, i + 1)

# Cargar y combinar resultados de todos los checkpoints
all_fitness_from_checkpoints = []
for i in range(NUMBER_EXPERIMENTS):
    checkpoint_file = f"checkpoint_{i}.pkl"
    if os.path.exists(checkpoint_file):
        checkpoint = load_checkpoint(checkpoint_file)
        all_fitness_from_checkpoints.append(checkpoint['best_fitness_per_gen'])

# Convertir los valores de fitness en una matriz para las gráficas
max_generations = max(len(fitness) for fitness in all_fitness_from_checkpoints)
generations = np.arange(1, max_generations + 1)
fitness_matrix = np.full((len(all_fitness_from_checkpoints), max_generations), np.nan)

for i, fitness in enumerate(all_fitness_from_checkpoints):
    fitness_matrix[i, :len(fitness)] = fitness

# Graficar el progreso
plt.figure(figsize=(12, 8))
for i in range(fitness_matrix.shape[0]):
    plt.plot(
        generations,
        fitness_matrix[i],
        linestyle='-',
        color='red',
        alpha=0.5
    )

# Calcular estadísticas
mean_fitness = np.nanmean(fitness_matrix, axis=0)
std_fitness = np.nanstd(fitness_matrix, axis=0)

# Graficar la media del fitness
plt.plot(
    generations,
    mean_fitness,
    linestyle='-',
    color='blue',
    label='Mean Fitness'
)

# Rellenar entre la media y la desviación estándar
plt.fill_between(
    generations,
    mean_fitness - std_fitness,
    mean_fitness + std_fitness,
    color='blue',
    alpha=0.2,
    label='Standard Deviation'
)

plt.title('Convergence Plot of Fitness per Generation')
plt.xlabel('Generations')
plt.ylabel('Fitness')
plt.legend()
plt.grid(True)
plt.show()

# Graficar Box Plot
plt.figure(figsize=(12, 8))
plt.boxplot(
    fitness_matrix.T,  # Transponer para que cada generación sea una caja
    patch_artist=True,
    showmeans=True,
    boxprops=dict(facecolor='lightblue', color='blue'),
    meanprops=dict(marker='o', markerfacecolor='red', markersize=5),
    medianprops=dict(color='green')
)
plt.title('Box Plot of Fitness per Generation')
plt.xlabel('Generations')
plt.ylabel('Fitness')
plt.grid(True)
plt.show()



Supracheckpoint cargado desde super_checkpoint.pkl


Running Experiments:   0%|          | 0/28 [00:00<?, ?it/s]

Checkpoint cargado desde checkpoint_2.pkl


Running Experiments:   4%|▎         | 1/28 [21:57<9:53:00, 1317.80s/it]

Supracheckpoint guardado en super_checkpoint.pkl
No se encontró un checkpoint previo en checkpoint_3.pkl. Comenzando desde cero.


Running Experiments:   7%|▋         | 2/28 [45:23<9:53:32, 1369.73s/it]

Supracheckpoint guardado en super_checkpoint.pkl
No se encontró un checkpoint previo en checkpoint_4.pkl. Comenzando desde cero.


Running Experiments:   7%|▋         | 2/28 [46:26<10:03:50, 1393.49s/it]


IndexError: list index out of range

In [ ]:


import os

def delete_checkpoints(folder_path, checkpoint_prefix="checkpoint_", super_checkpoint_file="super_checkpoint.pkl"):
    """
    Elimina todos los archivos de checkpoint y supracheckpoint.

    Args:
        folder_path (str): Ruta del directorio donde se encuentran los checkpoints.
        checkpoint_prefix (str): Prefijo de los archivos de checkpoint.
        super_checkpoint_file (str): Nombre del archivo de supracheckpoint.
    """
    # Borrar checkpoints individuales
    for filename in os.listdir(folder_path):
        if filename.startswith(checkpoint_prefix):
            file_path = os.path.join(folder_path, filename)
            try:
                os.remove(file_path)
                print(f"Eliminado: {file_path}")
            except Exception as e:
                print(f"Error al eliminar {file_path}: {e}")
    
    # Borrar supracheckpoint
    super_checkpoint_path = os.path.join(folder_path, super_checkpoint_file)
    if os.path.exists(super_checkpoint_path):
        try:
            os.remove(super_checkpoint_path)
            print(f"Eliminado: {super_checkpoint_path}")
        except Exception as e:
            print(f"Error al eliminar {super_checkpoint_path}: {e}")

# Ejecutar la función
delete_checkpoints(".", checkpoint_prefix="checkpoint_", super_checkpoint_file="super_checkpoint.pkl")


In [ ]:

# Cargar y combinar resultados de todos los checkpoints
all_fitness_from_checkpoints = []
best_individuals = []

for i in range(30):  # Ajusta el rango según el número de experimentos realizados
    checkpoint_file = f"checkpoint_{i}.pkl"
    if os.path.exists(checkpoint_file):
        checkpoint = load_checkpoint(checkpoint_file)
        all_fitness_from_checkpoints.append(checkpoint['best_fitness_per_gen'])
        best_individuals.append(max(checkpoint['population'], key=lambda x: x['fitness']))

# Encontrar el mejor individuo entre todos los experimentos
best_overall_individual = max(best_individuals, key=lambda x: x['fitness'])
print(f"El mejor individuo tiene un fitness de: {best_overall_individual['fitness']}")
print(f"Codificación del mejor individuo: {best_overall_individual['individual']}")




# Configuración de parámetros
class Config:
    def __init__(self, architecture='random', epochs=50, sample_rate=None, time=5, n_splits=5, window_size=5, checkpoint_file="training_checkpoint.json"):
        self.architecture = architecture
        self.epochs = epochs
        self.sample_rate = sample_rate
        self.time = time
        self.n_splits = n_splits
        self.window_size = window_size
        self.checkpoint_file = checkpoint_file

# Crear o cargar checkpoint
def load_checkpoint(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            checkpoint = json.load(f)
            print(f"Checkpoint cargado: {checkpoint}")
            return checkpoint
    return {"last_completed": -1}

def save_checkpoint(file_path, architecture_index):
    checkpoint = {"last_completed": architecture_index}
    with open(file_path, 'w') as f:
        json.dump(checkpoint, f)
    print(f"Checkpoint guardado: {checkpoint}")

# Split de audio en train y test
def train_test_split_audio(audio_dict):
    df = pd.read_csv('Dataset.csv', usecols=['Participant_ID', 'PHQ-9 Score'], dtype={1: str})
    df['labels'] = np.zeros([len(df),], dtype=int)
    df.loc[df['PHQ-9 Score'] < 10, 'labels'] = 0
    df.loc[df['PHQ-9 Score'] >= 10, 'labels'] = 1

    labels = df.set_index('Participant_ID').to_dict()['labels']

    X, Y = [], []
    for filename, data in tqdm(audio_dict.items(), 'LABEL'):
        ID = filename[:3]
        if ID in labels:
            dep = 0 if labels[ID] == 0 else 1
            [X.append(x) for x in data]
            [Y.append(dep) for x in data]

    X = pad_and_crop_spectrograms(X)
    Y = np.array(Y)

    X = X[..., np.newaxis]
    print(f"X shape: {X.shape}, Y shape: {Y.shape}")
    return X, Y

# Guardar resultados en CSV (append)
def append_results_to_csv(file_path, model_results):
    columns = ["Encoded Architecture", "Loss", "Accuracy", "Precision", "Recall", "F1", "Specificity"]

    # Convertir arquitectura a cadena para guardar
    model_results = [str(model_results[0])] + model_results[1:]

    # Verificar si el archivo ya existe
    if not os.path.exists(file_path):
        # Crear archivo con encabezados si no existe
        with open(file_path, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(columns)

    # Escribir resultados en el archivo
    with open(file_path, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(model_results)
    print(f"Resultados guardados en: {file_path}")

# Función de especificidad
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

# Entrenar y evaluar modelo
def train_and_evaluate_model(model, X_train, Y_train, X_val, Y_val, X_test, Y_test, config):
    model.compile(optimizer='adadelta', loss='binary_crossentropy', metrics=["accuracy", 'Precision', 'Recall'])
    model.fit(X_train, Y_train, epochs=config.epochs, validation_data=(X_val, Y_val), verbose=0)
    results = model.evaluate(X_test, Y_test, verbose=0)

    # Obtener predicciones para métricas adicionales
    Y_pred = (model.predict(X_test) > 0.5).astype("int32")
    accuracy = results[1]
    precision = precision_score(Y_test, Y_pred)
    recall = recall_score(Y_test, Y_pred)
    f1 = f1_score(Y_test, Y_pred)
    specificity = specificity_score(Y_test, Y_pred)

    return [results[0], accuracy, precision, recall, f1, specificity]

# Evaluar y almacenar resultados
def evaluate_and_store_model(architecture, X_train_val, X_test, Y_train_val, Y_test, config, use_kfold, stratified_kfold, target_shape, results_file):
    repaired_architecture = fixArch(architecture)
    decoded_model_dict = decode_model_architecture(repaired_architecture)
    model_results = [repaired_architecture]

    if use_kfold:
        fold_results = []
        for fold, (train_index, val_index) in enumerate(stratified_kfold.split(X_train_val, Y_train_val)):
            print(f"Entrenando fold {fold + 1}/{config.n_splits}...")
            X_train, X_val = X_train_val[train_index], X_train_val[val_index]
            Y_train, Y_val = Y_train_val[train_index], Y_train_val[val_index]
            tf_model = BuildPyTorchModel(decoded_model_dict, input_shape=(target_shape[0], target_shape[1], 1))
            fold_results.append(train_and_evaluate_model(tf_model, X_train, Y_train, X_val, Y_val, X_test, Y_test, config))
            print(f"Fold {fold + 1} completado.")

        avg_results = np.mean(fold_results, axis=0)
        model_results.extend(avg_results)

    else:
        X_train, X_val, Y_train, Y_val = train_test_split(X_train_val, Y_train_val, test_size=0.2, random_state=42)
        tf_model = BuildPyTorchModel(decoded_model_dict, input_shape=(target_shape[0], target_shape[1], 1))
        single_run_results = train_and_evaluate_model(tf_model, X_train, Y_train, X_val, Y_val, X_test, Y_test, config)
        model_results.extend(single_run_results)
        print("Modelo evaluado sin K-Fold Cross Validation.")

    # Guardar métricas de la arquitectura actual en el archivo CSV
    append_results_to_csv(results_file, model_results)

# Función de especificidad
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

# Entrenar y evaluar modelo
def train_and_evaluate_model(model, X_train, Y_train, X_val, Y_val, X_test, Y_test, epochs):
    model.compile(optimizer='adadelta', loss='binary_crossentropy', metrics=["accuracy", 'Precision', 'Recall'])
    model.fit(X_train, Y_train, epochs=epochs, validation_data=(X_val, Y_val), verbose=0)
    results = model.evaluate(X_test, Y_test, verbose=0)

    # Obtener predicciones para métricas adicionales
    Y_pred = (model.predict(X_test) > 0.5).astype("int32")
    accuracy = results[1]
    precision = precision_score(Y_test, Y_pred)
    recall = recall_score(Y_test, Y_pred)
    f1 = f1_score(Y_test, Y_pred)
    specificity = specificity_score(Y_test, Y_pred)

    return [results[0], accuracy, precision, recall, f1, specificity]

# Evaluar y almacenar resultados
def evaluate_best_individual(best_individual, X_train_val, X_test, Y_train_val, Y_test, target_shape, epochs=50, n_splits=5):
    # Decodificar la arquitectura
    repaired_architecture = fixArch(best_individual)
    decoded_model_dict = decode_model_architecture(repaired_architecture)

    fold_results = []
    stratified_kfold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for fold, (train_index, val_index) in enumerate(stratified_kfold.split(X_train_val, Y_train_val)):
        print(f"Entrenando fold {fold + 1}/{n_splits}...")
        X_train, X_val = X_train_val[train_index], X_train_val[val_index]
        Y_train, Y_val = Y_train_val[train_index], Y_train_val[val_index]
        
        # Crear el modelo a partir de la arquitectura
        tf_model = BuildPyTorchModel(decoded_model_dict, input_shape=(target_shape[0], target_shape[1], 1))
        fold_results.append(train_and_evaluate_model(tf_model, X_train, Y_train, X_val, Y_val, X_test, Y_test, epochs))
        print(f"Fold {fold + 1} completado.")

    avg_results = np.mean(fold_results, axis=0)
    print(f"Resultados promedio en {n_splits} folds: Loss={avg_results[0]:.4f}, Accuracy={avg_results[1]:.4f}, Precision={avg_results[2]:.4f}, Recall={avg_results[3]:.4f}, F1={avg_results[4]:.4f}, Specificity={avg_results[5]:.4f}")



# Cargar datos
directory = './SM-27'  # Ruta a los archivos de audio
window_size = 5  # Tamaño de la ventana en segundos
sample_rate = None  # Se determinará en la carga

print("Cargando y preprocesando datos de audio...")
audio_dict, sample_rate = load_audio_data(directory, window_size, sample_rate)
audio_dict = preprocess_audio(audio_dict, sample_rate)
X, Y = train_test_split_audio(audio_dict)

X_train_val, X_test, Y_train_val, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Evaluar el mejor individuo (sustituye 'best_overall_individual' con la arquitectura específica)
evaluate_best_individual(
    best_overall_individual['individual'],
    X_train_val,
    X_test,
    Y_train_val,
    Y_test,
    target_shape=(128, 128),
    epochs=100,
    n_splits=5
)


best_individual = best_overall_individual['individual']
constructed_model = BuildPyTorchModel(decode_model_architecture(best_individual))
print(constructed_model.summary())




